# 01 — Normalize Fetched Submissions
Transforms `pickles/fetched_submissions.pkl` (raw Arctic Shift API data) into format expected by the rest of the pipeline.

In [1]:
from datetime import datetime
import warnings
import redditcleaner
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Config

In [2]:
INPUT_FILE  = Path("pickles/fetched_submissions.pkl")
OUTPUT_FILE = Path("pickles/first-date_posts-all.pkl")

# Columns expected by the downstream pipeline (matches 00-dataset_preparation output)
OUTPUT_COLUMNS = [
    "author",
    "created",
    "created_utc",
    "id",
    "num_comments",
    "score",
    "selftext",
    "title",
    "subreddit",
    "subreddit_id",
    "url",
    "extraction_keywords",
    "subreddit_subscribers",
]

## Load

In [12]:
df = pd.read_pickle(INPUT_FILE)
print(f"Loaded: {df.shape}")

Loaded: (87154, 125)


## Filter (mirrors `filter_posts` in 00-dataset_preparation)

In [13]:
n0 = len(df)

# Drop empty / removed / deleted bodies
df = df[df["selftext"].notna()]
df = df[~df["selftext"].isin(["", "[removed]", "[deleted]"])]
print(f"After removing empty/removed/deleted: {len(df):,} (dropped {n0 - len(df):,})")

# Drop exact-text duplicates
n1 = len(df)
df = df[~df.duplicated(subset=["selftext"], keep=False)]
print(f"After dedup: {len(df):,} (dropped {n1 - len(df):,})")

# Drop removed posts
n2 = len(df)
if "removed_by_category" in df.columns:
    df = df[df["removed_by_category"].isna()]
print(f"After removed_by_category: {len(df):,} (dropped {n2 - len(df):,})")

# Drop locked posts
n3 = len(df)
if "locked" in df.columns:
    df = df[df["locked"] != True]
print(f"After locked: {len(df):,} (dropped {n3 - len(df):,})")

# Drop moderator-distinguished posts
n4 = len(df)
if "distinguished" in df.columns:
    df = df[df["distinguished"] != "moderator"]
print(f"After distinguished: {len(df):,} (dropped {n4 - len(df):,})")

print(f"\nTotal kept: {len(df):,} / {n0:,}")

After removing empty/removed/deleted: 87,154 (dropped 0)
After dedup: 84,274 (dropped 2,880)
After removed_by_category: 84,274 (dropped 0)
After locked: 84,273 (dropped 1)
After distinguished: 84,273 (dropped 0)

Total kept: 84,273 / 87,154


## Clean text & derive columns

In [14]:
df["selftext"] = df["selftext"].apply(redditcleaner.clean)
df["title"]    = df["title"].apply(redditcleaner.clean)

df["selftext"] = df["selftext"].str.replace("&amp;", "and", regex=False)
df["title"]    = df["title"].str.replace("&amp;", "and", regex=False)

df["created"] = df["created_utc"].apply(
    lambda x: datetime.fromtimestamp(int(x)) if pd.notna(x) else pd.NA
)

# Re-derive extraction_keywords by scanning text — mirrors filter_posts_by_keywords
KEYWORDS = [
    "first date",
    "1st date",
    "first time meeting",
    "date zero",
    "met IRL",
]

def find_keywords(row: pd.Series) -> list[str]:
    text = f"{row['title']} {row['selftext']}".lower()
    return [kw for kw in KEYWORDS if kw.lower() in text]

df["extraction_keywords"] = df.apply(find_keywords, axis=1)

df = df.sort_values(by="created", ascending=False)

## Select output columns & save

In [15]:
missing_cols = [c for c in OUTPUT_COLUMNS if c not in df.columns]
if missing_cols:
    print(f"WARNING: missing columns (will be NaN): {missing_cols}")
    for c in missing_cols:
        df[c] = None

df = df[OUTPUT_COLUMNS]
print(f"Final shape: {df.shape}")
df.head(2)

Final shape: (84273, 13)


,author,created,created_utc,id,num_comments,score,selftext,title,subreddit,subreddit_id,url,extraction_keywords,subreddit_subscribers
36761,tehkobalt,2026-01-10 05:38:35,1768019915,1q8urb3,7,2,"I really dont get it - we talk for a few, I ma...",How do you actually online date? M26 Australian,OnlineDating,t5_2qpe9,https://www.reddit.com/r/OnlineDating/comments...,[first date],202611.0
18118,stevew25,2026-01-09 04:46:24,1767930384,1q7xpie,16,2,I have a like and she is very attractive but a...,Should I match with someone who is attractive ...,OnlineDating,t5_2qpe9,https://www.reddit.com/r/OnlineDating/comments...,[first date],201872.0


In [25]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_pickle(OUTPUT_FILE)
print(f"Saved → {OUTPUT_FILE}")

Saved → pickles/first-date_posts-all.pkl
